# Comparación: caso base

¿Ayuda que Dreamer explore al azar en el ambiente de la galleta? Para saberlo se corren dos notebooks **idénticos salvo en una cosa**: quién elige las acciones al juntar datos.

**Este notebook es el caso base**: Dreamer junta los datos con la política que va aprendiendo, como siempre. El otro es `comparacion_random.ipynb`; córrelo también, en otra sesión de Colab o después de este.

Lo que se compara es `episode/eval_score`: cada 10.000 pasos se juegan 10 episodios con el actor aprendido (sin azar) y se promedian las galletas. En los dos casos se mide igual. El puntaje de entrenamiento (`episode/score`) **no sirve para comparar**, porque en el caso aleatorio es el de un agente al azar.

El ambiente es el mismo en los dos: el agente solo gira y avanza, y avanzar contra el botón lo aprieta (`env.press_on_move=True`, el cambio que sugirió el profe).

Si la sesión se corta, vuelve a correr las celdas 1 a 6 y después la 9: el run se retoma solo desde el último checkpoint en Drive.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.

## 1. Revisar la GPU que tocó

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Montar Drive

Aquí van los checkpoints y las métricas, para que sobrevivan al corte de sesión y para poder comparar los dos casos al final.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clonar el repo (o actualizarlo si ya está)

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 4. Instalar

r2dreamer necesita Python 3.11 y Colab suele traer una versión más nueva, así que se crea un entorno aparte con `uv`. Toma unos minutos y hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
pip install -q uv
cd /content/r2dreamer-cookie
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

## 5. Comprobar que todo quedó bien

In [ ]:
%%bash
cd /content/r2dreamer-cookie
.venv/bin/python - <<'EOF'
import torch, gymnasium, cookie_env
from envs.cookie import Cookie
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
a = torch.randn(256, 256, device="cuda", dtype=torch.float16)
print("float16 ok:", bool(torch.isfinite((a @ a).sum())))
env = Cookie("partial")
print("galleta ok:", env.reset()["image"].shape, "| acciones:", env.action_space.n, "(deberían ser 3)")
EOF

## 6. Configurar el run

**No cambies `CASO`**: es lo que distingue este notebook del otro.

Si cambias cualquier otra cosa (`TAREA`, `PASOS`, `TRAIN_RATIO`, `SEMILLA`, `REP_LOSS`), **cámbiala igual en el otro notebook**, o la comparación deja de ser justa.

- `TAREA`: se parte con `cookie_fullfixed` (observación completa, la galleta no reaparece), donde según Nano Dreamer sí aprende. Cuando eso funcione, se repite con `cookie_partial`, que es el caso que falla.
- `TRAIN_RATIO`: 128 corre unas 4 veces más rápido que el 512 original, a cambio de menos actualizaciones por dato. Sirve para estas primeras pruebas.
- `SEMILLA`: con recompensas tan escasas, una sola corrida puede engañar. Si hay tiempo, repite ambos casos con `SEMILLA = 1` y `2`; cada una queda en su propia carpeta.

In [ ]:
CASO = "base"  # no cambiar

TAREA = "cookie_fullfixed"
PASOS = 100_000
TRAIN_RATIO = 128
SEMILLA = 0
REP_LOSS = "r2dreamer"

LOGDIR = f"/content/drive/MyDrive/galleta/comparacion/{TAREA}_s{SEMILLA}_{CASO}"

OPCIONES = (
    f"env=cookie env.task={TAREA} env.train_ratio={TRAIN_RATIO} "
    f"seed={SEMILLA} model.rep_loss={REP_LOSS} "
    f"trainer.random_policy={CASO == 'random'}"
)
print(LOGDIR)
print(OPCIONES)

## 7. Prueba corta (unos 10 minutos)

Antes del run largo: corre 1500 pasos en disco local (no en Drive), con un buffer chico que no se guarda y un solo episodio de evaluación. Comprueba que todo funcione y estima cuánto va a tardar el run completo.

In [ ]:
!rm -rf /content/prueba_comparacion
!cd /content/r2dreamer-cookie && timeout 1200 .venv/bin/python -u train.py {OPCIONES} \
    env.steps=1500 env.eval_episode_num=1 trainer.update_log_every=500 trainer.save_every=1e9 \
    buffer.max_size=1e4 trainer.save_buffer=False \
    logdir=/content/prueba_comparacion > /content/prueba_comparacion.log 2>&1; \
    tail -n 5 /content/prueba_comparacion.log

In [ ]:
import json

lineas = [json.loads(l) for l in open("/content/prueba_comparacion/metrics.jsonl")]
fps = [l["fps/fps"] for l in lineas if l.get("fps/fps", 0) > 0]
evaluo = any("episode/eval_score" in l for l in lineas)
print("evaluación ok:", evaluo)
if not fps:
    print("No alcanzó a medir la velocidad. Revisa el final de /content/prueba_comparacion.log (arriba).")
else:
    v = sum(fps) / len(fps)
    print(f"{v:.1f} pasos por segundo → {PASOS:,} pasos en unas {PASOS / v / 3600:.1f} horas, "
          "más unos minutos por cada evaluación")

## 8. Abrir TensorBoard

Ábrelo **antes** de entrenar: mientras la celda de entrenamiento corre, Colab no deja ejecutar otras, pero TensorBoard se sigue actualizando (botón de recargar arriba a la derecha).

Muestra las carpetas de los dos casos juntas. Lo que importa está en la pestaña Scalars:

- `episode/eval_score`: **la comparación**. Galletas por episodio del actor aprendido.
- `train/loss/rew`: si el modelo aprende dónde está la recompensa.
- `episode/score`: galletas mientras junta datos. En el caso aleatorio es el de un agente al azar (unas 0,05 por episodio).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/galleta/comparacion"

## 9. Entrenar

Abajo aparece una barra de progreso como esta:

```
 17%|#########8                                     | 17000/100000 [25:10<2:02:53, 11.3step/s]
```

Se lee así: pasos hechos / total, `[tiempo transcurrido<tiempo que falta, velocidad]`. El tiempo que falta se calcula con la velocidad promedio de todo el run, evaluaciones incluidas, así que se va afinando a medida que avanza. Durante las evaluaciones la barra se detiene unos minutos y dice `evaluating`. Al principio también se queda quieta unos minutos mientras compila el modelo; es normal.

Encima de la barra van quedando las líneas del log: una por episodio (`episode/score`), una cada 2500 pasos con las pérdidas y una por evaluación (`episode/eval_score`).

Para parar antes, usa el botón de detener de la celda: el último checkpoint queda en Drive.

In [ ]:
!cd /content/r2dreamer-cookie && .venv/bin/python -u train.py {OPCIONES} \
    env.steps={PASOS} trainer.update_log_every=2500 trainer.save_every=1e4 buffer.max_size=1e5 \
    logdir="{LOGDIR}"

## 10. Comparar los dos casos

Córrelo cuando **los dos** notebooks hayan terminado (o para ver cómo van). Lee las métricas de Drive y grafica el `eval_score` de ambos en función de los pasos. La línea punteada es un agente al azar, como referencia: si una curva no la supera, ese caso no aprendió nada.

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

AZAR = 0.05  # galletas por episodio de un agente al azar con 3 acciones (simulación de 200 episodios)

fig, ax = plt.subplots(figsize=(7, 4))
for caso, color in (("base", "tab:blue"), ("random", "tab:orange")):
    archivo = pathlib.Path(f"/content/drive/MyDrive/galleta/comparacion/{TAREA}_s{SEMILLA}_{caso}/metrics.jsonl")
    if not archivo.exists():
        print(f"Todavía no hay métricas de '{caso}' en {archivo.parent}")
        continue
    puntos = [json.loads(l) for l in open(archivo)]
    puntos = [(p["step"], p["episode/eval_score"]) for p in puntos if "episode/eval_score" in p]
    if puntos:
        pasos, puntaje = zip(*puntos)
        ax.plot(pasos, puntaje, marker="o", color=color, label=caso)
        print(f"{caso}: última evaluación en el paso {pasos[-1]:,} → {puntaje[-1]:.2f} galletas por episodio")
ax.axhline(AZAR, color="gray", linestyle="--", label="agente al azar")
ax.set_xlabel("pasos del ambiente")
ax.set_ylabel("galletas por episodio (evaluación)")
ax.set_title(f"{TAREA}, semilla {SEMILLA}")
ax.legend()
plt.show()